# PYNQ-Z1 MAC AXI4-Lite Demo

這個 Notebook 會載入 `mac_npu.bit`，透過 AXI4-Lite MMIO 寫入 signed INT8 操作數，並讀回 signed INT32 accumulator。

暫存器配置：`0x00 CONTROL`、`0x04 A`、`0x08 B`、`0x0C STATUS`、`0x10 ACCUMULATOR`。

## 1. 找到 bitstream 並載入驅動

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("XILINX_XRT", "/usr")

candidates = [
    Path.cwd().parents[1] / "overlay" / "mac_npu.bit",
    Path.cwd().parent / "overlay" / "mac_npu.bit",
    Path.cwd() / "overlay" / "mac_npu.bit",
    Path("/home/xilinx/jupyter_notebooks/pynq_z1_repo/mac_npu/overlay/mac_npu.bit"),
]
BITFILE = next((path.resolve() for path in candidates if path.exists()), None)
assert BITFILE is not None, "找不到 mac_npu.bit，請確認專案已上傳到 PYNQ。"
assert BITFILE.with_suffix(".hwh").exists(), "mac_npu.hwh 必須和 bitstream 放在同一目錄。"

PROJECT_DIR = BITFILE.parent.parent
SERVICE_DIR = PROJECT_DIR / "core" / "service"
sys.path.insert(0, str(SERVICE_DIR))

from mac_mmio import load_mac_overlay

print(f"Bitstream: {BITFILE}")

## 2. Program FPGA 並建立 MMIO 控制器

In [ ]:
overlay, mac = load_mac_overlay(BITFILE, ip_name="mac_axi_lite_0")
ip = overlay.ip_dict["mac_axi_lite_0"]
print("Overlay loaded")
print(f"Base address: 0x{ip.get('phys_addr', ip.get('base_address')):08X}")
print(f"Address range: 0x{ip.get('addr_range', ip.get('range')):X}")

## 3. Clear accumulator

In [ ]:
mac.clear()
accumulator = mac.read_accumulator()
assert accumulator == 0
print(f"Accumulator after clear: {accumulator}")

## 4. 執行 signed INT8 MAC

每次呼叫 `mac(a, b)` 都會計算 `accumulator = accumulator + a × b`。

In [ ]:
vectors = [
    (2, 3),
    (-7, 6),
    (127, -1),
]

expected = 0
for a, b in vectors:
    expected += a * b
    hardware_result = mac.mac(a, b)
    assert hardware_result == expected
    print(f"{a:4d} × {b:4d} -> accumulator = {hardware_result:6d}")

## 5. 邊界值與完整自動驗證

In [ ]:
mac.clear()
checks = [
    (-128, -128, 16384),
    (127, -128, 128),
]

for a, b, expected_accumulator in checks:
    actual = mac.mac(a, b)
    assert actual == expected_accumulator, (a, b, expected_accumulator, actual)

print("PASS: bitstream load, AXI4-Lite MMIO, signed MAC and accumulation verified")

## 自訂測試

操作數必須介於 `-128` 到 `127`。修改下方的 `a`、`b` 後重新執行即可。

In [ ]:
mac.clear()
a, b = 12, -5
result = mac.mac(a, b)
print(f"{a} × {b} = {result}")